<a href="https://colab.research.google.com/github/malikasadnadir-max/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/malikasadnadir-max/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [2]:
# ML-10 setup: restore repository if needed

from pathlib import Path
import subprocess

repo_path = Path("/content/flyrank-ml-internship")

if not repo_path.exists():
    print("Repository not found. Cloning...")
    subprocess.run(
        [
            "git",
            "clone",
            "https://github.com/malikasadnadir-max/flyrank-ml-internship.git",
            str(repo_path)
        ],
        check=True
    )
else:
    print("Repository already exists.")

data_path = repo_path / "data/raw/content_refresh_anonymized.csv"

print("\nRepository exists:", repo_path.exists())
print("Dataset exists:", data_path.exists())
print("Dataset path:", data_path)

if not data_path.exists():
    raise FileNotFoundError(
        "The repository was restored, but the dataset is not present."
    )

print("\nSetup check: PASS")

Repository not found. Cloning...

Repository exists: True
Dataset exists: True
Dataset path: /content/flyrank-ml-internship/data/raw/content_refresh_anonymized.csv

Setup check: PASS


## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*

## 1. Ranked Actions + Reason Codes

The playbook ranks pages for human review rather than automatically changing content.

The ranking uses the validated Logistic Regression model from Week 5, using the leakage-audited feature set:

* `impressions_90d`
* `clicks_90d`
* `ctr`
* `avg_position`
* `days_since_last_update`
* `content_age_days`
* `word_count`

The model score is used as a prioritization signal, not as a guarantee that a page will decline.

Each ranked page also receives a human-readable reason code:

* **STALE_REFRESH** — the page has not been updated recently and has sufficient search visibility to justify review.
* **LOW_CTR_POSITION** — the page has meaningful impressions, has a usable average position, and has relatively low CTR for its observed position.
* **STALE_AND_LOW_CTR** — both signals are present, so the page has two independent reasons for review.
* **MONITOR** — the page does not meet either action signal and should not be prioritized for immediate content changes.

The queue is therefore intended to answer: **which pages should a human review first, and what observable signal explains that priority?**


In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-10 Section 1: Build ranked action queue

import numpy as np
import pandas as pd

from pathlib import Path
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression

# ---------------------------------------------------------
# 1. Load dataset
# ---------------------------------------------------------

repo_path = Path("/content/flyrank-ml-internship")
data_path = repo_path / "data/raw/content_refresh_anonymized.csv"

if not data_path.exists():
    raise FileNotFoundError(f"Dataset not found: {data_path}")

df = pd.read_csv(data_path)

print("Dataset shape:", df.shape)

# ---------------------------------------------------------
# 2. Define final validated feature set
# ---------------------------------------------------------

model_features = [
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "days_since_last_update",
    "content_age_days",
    "word_count"
]

target_col = "is_declining_label"

df[target_col] = (
    df["trend_direction"]
    .astype(str)
    .str.lower()
    .eq("down")
    .astype(int)
)

# ---------------------------------------------------------
# 3. Train the same Logistic Regression model
# ---------------------------------------------------------

X = df[model_features].copy()
y = df[target_col].copy()

model = Pipeline([
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler()),
    ("logistic_regression", LogisticRegression(
        max_iter=1000,
        random_state=42
    ))
])

model.fit(X, y)

df["model_priority_score"] = model.predict_proba(X)[:, 1]

# ---------------------------------------------------------
# 4. Build transparent reason signals
# ---------------------------------------------------------

df["stale_signal"] = (
    df["days_since_last_update"] >= 180
)

df["low_ctr_position_signal"] = (
    (df["impressions_90d"] >= 500)
    & (df["avg_position"] > 0)
    & (df["avg_position"] <= 20)
    & (df["ctr"] < 0.5)
)

df["action_score"] = (
    2 * df["stale_signal"].astype(int)
    + df["low_ctr_position_signal"].astype(int)
)

def reason_code(row):
    if row["stale_signal"] and row["low_ctr_position_signal"]:
        return "STALE_AND_LOW_CTR"
    elif row["stale_signal"]:
        return "STALE_REFRESH"
    elif row["low_ctr_position_signal"]:
        return "LOW_CTR_POSITION"
    else:
        return "MONITOR"

df["reason_code"] = df.apply(reason_code, axis=1)

# ---------------------------------------------------------
# 5. Map reason codes to recommended actions
# ---------------------------------------------------------

action_map = {
    "STALE_AND_LOW_CTR": "Review freshness and CTR/position together",
    "STALE_REFRESH": "Review for content refresh",
    "LOW_CTR_POSITION": "Review title/snippet and search-intent alignment",
    "MONITOR": "Monitor; no immediate content change"
}

df["recommended_action"] = df["reason_code"].map(action_map)

# ---------------------------------------------------------
# 6. Rank
# ---------------------------------------------------------

queue = (
    df.sort_values(
        ["model_priority_score", "action_score"],
        ascending=[False, False]
    )
    .reset_index(drop=True)
)

queue["queue_rank"] = np.arange(1, len(queue) + 1)

# ---------------------------------------------------------
# 7. Display summary
# ---------------------------------------------------------

print("\nReason-code counts:")
print(queue["reason_code"].value_counts())

print("\nTop 20 action queue:")
print(
    queue[
        [
            "queue_rank",
            "model_priority_score",
            "action_score",
            "reason_code",
            "recommended_action"
        ]
    ].head(20).to_string(index=False)
)

print("\nSection 1 checks: PASS")

Dataset shape: (30000, 44)

Reason-code counts:
reason_code
MONITOR              20077
LOW_CTR_POSITION      9749
STALE_REFRESH          164
STALE_AND_LOW_CTR       10
Name: count, dtype: int64

Top 20 action queue:
 queue_rank  model_priority_score  action_score      reason_code                               recommended_action
          1              0.778606             2    STALE_REFRESH                       Review for content refresh
          2              0.775929             2    STALE_REFRESH                       Review for content refresh
          3              0.775793             2    STALE_REFRESH                       Review for content refresh
          4              0.775346             2    STALE_REFRESH                       Review for content refresh
          5              0.774257             2    STALE_REFRESH                       Review for content refresh
          6              0.771673             2    STALE_REFRESH                       Review for co

## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

## 2. Intended Use and Limits

### Intended use

The playbook is intended for content and SEO teams that need to prioritize which pages deserve human review first.

The model ranking provides decision-support by ordering pages according to their measured model score. Reason codes provide additional context so that a reviewer can understand why a page entered the queue.

The playbook is most useful when review capacity is limited and the team needs a repeatable way to prioritize pages.

### Limits

The model was validated with a grouped-by-client split. In that evaluation, the model achieved Precision@50 of 0.600 compared with 0.540 for the baseline. This is evidence about the measured held-out client split, not a guarantee of future performance.

The grouped validation tests generalization to unseen clients. It does not establish performance on a future time period.

The model also does not establish that refreshing a page will cause its performance to improve. The observed relationships are decision-support signals rather than causal recommendations.

The queue should therefore be treated as a prioritization tool for human review, not an autonomous content-optimization system.


In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-10 Section 2: Intended-use checks

validated_model_p50 = 0.600
validated_baseline_p50 = 0.540

validated_difference = (
    validated_model_p50 - validated_baseline_p50
)

print("Validated grouped-client Model Precision@50:",
      validated_model_p50)

print("Validated grouped-client Baseline Precision@50:",
      validated_baseline_p50)

print("Measured difference:",
      f"{validated_difference:+.3f}")

assert round(validated_difference, 3) == 0.060

assert "trend_direction" not in model_features
assert "trend_pct" not in model_features
assert "future_clicks" not in model_features
assert "future_impressions" not in model_features

print("\nSection 2 checks: PASS")

Validated grouped-client Model Precision@50: 0.6
Validated grouped-client Baseline Precision@50: 0.54
Measured difference: +0.060

Section 2 checks: PASS


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

## 3. Human Review + the No-Go List

Every page in the queue requires human review before an action is taken.

### Human review checklist

Before changing a page, the reviewer should check:

1. **Search intent** — Does the current page actually satisfy the intent behind the queries bringing impressions?
2. **Content quality** — Is the information accurate, useful, complete, and current?
3. **Freshness** — If the page is old, is there genuinely new information that justifies an update?
4. **CTR context** — If CTR is low, could SERP features, query intent, branding, or competition explain the result?
5. **Position context** — Confirm that `avg_position = 0` is treated as missing/no usable position data rather than as a ranking of zero.
6. **Business context** — Confirm that the proposed change is appropriate for the page's purpose and business constraints.
7. **Cost versus value** — Prioritize changes where the expected review effort is reasonable relative to the page's observed visibility and potential importance.

### No-go list

The system should never automatically:

* publish or rewrite page content;
* change titles or meta descriptions without review;
* delete pages;
* redirect URLs;
* make claims about Google's ranking algorithm;
* declare that a refresh will improve traffic;
* make decisions using client identifiers as model features;
* use future outcome fields to justify a current action;
* treat a model score as proof that a page is actually declining.

The output is a review queue, not an autonomous publishing system.


In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-10 Section 3: Human-review / no-go checks

required_reason_codes = {
    "STALE_AND_LOW_CTR",
    "STALE_REFRESH",
    "LOW_CTR_POSITION",
    "MONITOR"
}

actual_reason_codes = set(queue["reason_code"].dropna().unique())

assert actual_reason_codes.issubset(required_reason_codes)

assert "recommended_action" in queue.columns
assert "model_priority_score" in queue.columns
assert "reason_code" in queue.columns

# Model identifiers are not part of the feature set
assert "client_id" not in model_features
assert "content_id" not in model_features

print("Reason codes:", sorted(actual_reason_codes))
print("Human-review requirement: PASS")
print("No-go feature checks: PASS")

print("\nSection 3 checks: PASS")

Reason codes: ['LOW_CTR_POSITION', 'MONITOR', 'STALE_AND_LOW_CTR', 'STALE_REFRESH']
Human-review requirement: PASS
No-go feature checks: PASS

Section 3 checks: PASS


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

## 4. Monitoring / Retrain Triggers

The playbook should be monitored rather than assumed to remain valid indefinitely.

### Monitoring signals

I would monitor:

* Precision@50 on a later labeled evaluation period when a valid future label becomes available.
* The share of pages receiving each reason code.
* Missingness and distribution changes in the seven model features.
* Changes in the distribution of model priority scores.
* Whether human reviewers frequently reject the suggested reason or action.
* Whether the client mix changes substantially from the validation population.

### Retrain or review triggers

A model review should be triggered when:

1. measured Precision@50 falls materially below the previously observed 0.600 on a comparable evaluation design;
2. feature distributions or missingness change substantially;
3. the action queue becomes dominated by one reason code without a corresponding business explanation;
4. human reviewers repeatedly disagree with the ranking or reason codes;
5. the data-generating process or feature definitions change.

These are monitoring and review triggers, not guarantees that retraining will solve the underlying problem. The evaluation design should be checked first.


In [8]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-10 Section 4: Monitoring snapshot

print("Queue size:", len(queue))

print("\nReason-code distribution:")
reason_distribution = (
    queue["reason_code"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print(reason_distribution)

print("\nModel score summary:")
print(
    queue["model_priority_score"]
    .describe()
    .round(4)
)

print("\nFeature missingness (%):")
feature_missingness = (
    df[model_features]
    .isna()
    .mean()
    .mul(100)
    .round(2)
)

print(feature_missingness)

assert len(queue) == len(df)
assert queue["model_priority_score"].notna().all()
assert queue["reason_code"].notna().all()

print("\nSection 4 checks: PASS")

Queue size: 30000

Reason-code distribution:
reason_code
MONITOR              66.92
LOW_CTR_POSITION     32.50
STALE_REFRESH         0.55
STALE_AND_LOW_CTR     0.03
Name: proportion, dtype: float64

Model score summary:
count    30000.0000
mean         0.5421
std          0.1032
min          0.0003
25%          0.4649
50%          0.5766
75%          0.6146
max          0.7786
Name: model_priority_score, dtype: float64

Feature missingness (%):
impressions_90d            0.00
clicks_90d                 0.00
ctr                        0.00
avg_position               0.00
days_since_last_update     0.00
content_age_days           0.00
word_count                25.66
dtype: float64

Section 4 checks: PASS


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*

## 5. Exports for the Paper

The ranked action queue is exported to `work/outputs/` so that the research paper can reuse the exact queue generated by this notebook.

The export contains the model priority score, action score, reason code, recommended action, and supporting feature values needed for human review.

The CSV is generated by the notebook and should remain excluded from Git according to the repository's data-safety and CI rules. The notebook itself is the reproducible source for regenerating the queue.


In [9]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# ML-10 Section 5: Export ranked queue for the paper

from pathlib import Path

output_dir = Path("/content/flyrank-ml-internship/work/outputs")
output_dir.mkdir(parents=True, exist_ok=True)

queue_output = output_dir / "ml10_ranked_action_queue.csv"

export_columns = [
    "queue_rank",
    "model_priority_score",
    "action_score",
    "reason_code",
    "recommended_action",
    "impressions_90d",
    "clicks_90d",
    "ctr",
    "avg_position",
    "days_since_last_update",
    "content_age_days",
    "word_count"
]

queue_export = queue[export_columns].copy()

queue_export.to_csv(queue_output, index=False)

print("Exported:")
print(queue_output)

print("\nRows exported:", len(queue_export))
print("Columns exported:", len(queue_export.columns))

assert queue_output.exists()
assert len(queue_export) == len(queue)

print("\nSection 5 checks: PASS")

Exported:
/content/flyrank-ml-internship/work/outputs/ml10_ranked_action_queue.csv

Rows exported: 30000
Columns exported: 12

Section 5 checks: PASS


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.